In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/02/26 14:46:09 WARN Utils: Your hostname, bsudell resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/26 14:46:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/02/26 14:46:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [24]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [6]:
df_repartitioned = df.repartition(4)

df_repartitioned.write.mode("overwrite").parquet("yellow_nov_2025_repartitioned")

In [25]:
df_15 = df.filter("tpep_pickup_datetime >= '2025-11-15 00:00:00' AND tpep_pickup_datetime < '2025-11-16 00:00:00'")
df_15.count()

160466

In [20]:
from pyspark.sql import functions as F

print(f"Total rows: {df.count()}")

count_a = df.filter(F.to_date("tpep_pickup_datetime") == "2025-11-15").count()

count_b = df.filter(
    (F.col("tpep_pickup_datetime") >= "2025-11-15 00:00:00") & 
    (F.col("tpep_pickup_datetime") < "2025-11-16 00:00:00")
).count()

print(f"Count A: {count_a}")
print(f"Count B: {count_b}")

Total rows: 4181444
Count A: 160466
Count B: 160466


In [31]:
df_duration = df.withColumn("duration_hours", 
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

max_duration = df_duration.select(F.max("duration_hours")).collect()[0][0]

print(f"The longest trip is: {max_duration} hours")

The longest trip is: 90.64666666666666 hours


In [32]:
zone_df = spark.read.csv(
    "taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

In [34]:
df.createOrReplaceTempView("trips")
zone_df.createOrReplaceTempView("zones")

In [35]:
result = spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(1) as total_trips
    FROM 
        trips t
    JOIN 
        zones z ON t.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
    ORDER BY 
        total_trips ASC
    LIMIT 5
""")

In [36]:
result.show(truncate=False)

+---------------------------------------------+-----------+
|Zone                                         |total_trips|
+---------------------------------------------+-----------+
|Governor's Island/Ellis Island/Liberty Island|1          |
|Eltingville/Annadale/Prince's Bay            |1          |
|Arden Heights                                |1          |
|Port Richmond                                |3          |
|Rikers Island                                |4          |
+---------------------------------------------+-----------+

